# DTL: classify probability map

In [ ]:
# CONFIG

VERSION = 'v6'

# 1. Load probability stack asset from step 3

In [ ]:
!python -m pip install .. --quiet

import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')

# Load probability stack from asset

prob_stack = ee.Image(f'projects/epistem2/assets/probability_stack_Aceh_2020_{VERSION}')

provinces = ee.FeatureCollection('projects/epistem2/assets/AOI_Sumatra_Provinces')
province_list = provinces.toList(provinces.size())
aoi = ee.Feature(province_list.get(0)).geometry()

stacked_landsat = ee.Image(f'projects/epistem2/assets/stacked_landsat_2020_sumatra_{VERSION}')
band_names = stacked_landsat.bandNames()

# 2. Classify the prob maps

Using `Argmax`

In [ ]:
# Derive class_ids directly from prob_stack's band names — guaranteed to match
prob_band_names = prob_stack.bandNames().getInfo()  # e.g. ['prob_18', 'prob_2', ...]

# Extract the numeric class id from each band name, keep prob_stack's own order
class_ids_list = [int(b.split('_')[1]) for b in prob_band_names]
class_ids = ee.List(class_ids_list)

print("class_ids (from prob_stack):", class_ids_list)

# prob_stack is already in this exact order — no need to reorder/select
max_prob_index = prob_stack.toArray().arrayArgmax().arrayGet(0)

final_lc = max_prob_index.remap(
    ee.List.sequence(0, class_ids.size().subtract(1)),
    class_ids
).rename('classification')

max_confidence = prob_stack.toArray().arrayReduce(ee.Reducer.max(), [0]).arrayGet([0]).rename('confidence')

## Optional: visualization and statistic check

In [ ]:
import geemap

Map = geemap.Map()
Map.centerObject(aoi, 9)

class_ids_list = class_ids.getInfo()
class_palette = ['e31a1c','33a02c','1f78b4','ff7f00','b15928',
                  '6a3d9a','a6cee3','b2df8a','fb9a99','fdbf6f','cab2d6']

n_classes = class_ids.size().getInfo()

classification_vis = {
    'min': min(class_ids_list),
    'max': max(class_ids_list),
    'palette': class_palette[:len(class_ids_list)]
}


confidence_vis = {
    'min': 0,
    'max': 100,
    'palette': ['ffffcc', 'ffeda0', 'fed976', 'feb24c', 'fd8d3c', 'fc4e2a', 'e31a1c', 'b10026']
}

Map.addLayer(aoi, {}, 'AOI', True)
Map.addLayer(final_lc.clip(aoi), classification_vis, 'Final Classification')
Map.addLayer(max_confidence.clip(aoi), confidence_vis, 'Confidence', False)

# Map.add_legend(
#     title="Land Cover Class",
#     labels=[str(c) for c in class_ids.getInfo()],
#     colors=class_palette[:n_classes]
# )

# Map

In [ ]:
# %%
# Compute area per class using pixelArea + group reducer
area_image = ee.Image.pixelArea().addBands(final_lc)

area_stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName='class'
    ),
    geometry=aoi,
    scale=100,
    maxPixels=1e13,
    bestEffort=True
)

area_stats_info = area_stats.getInfo()
print(area_stats_info)

# %%
import pandas as pd

# Parse into a clean DataFrame
groups = area_stats_info['groups']

df_stats = pd.DataFrame(groups)
df_stats = df_stats.rename(columns={'class': 'class_id', 'sum': 'area_m2'})
df_stats['area_ha'] = df_stats['area_m2'] / 10_000
df_stats['area_km2'] = df_stats['area_m2'] / 1_000_000
df_stats['pct_of_total'] = (df_stats['area_m2'] / df_stats['area_m2'].sum()) * 100

df_stats = df_stats.sort_values('area_ha', ascending=False).reset_index(drop=True)
df_stats

# %%
# Pixel counts per class (useful sanity check alongside area)
pixel_count_stats = final_lc.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=100,
    maxPixels=1e13,
    bestEffort=True
).getInfo()

pixel_counts = pixel_count_stats['classification']
df_pixels = pd.DataFrame(
    [{'class_id': int(k), 'pixel_count': v} for k, v in pixel_counts.items()]
).sort_values('pixel_count', ascending=False).reset_index(drop=True)

df_pixels

# %%
# Merge for a single summary table
df_summary = df_stats.merge(df_pixels, on='class_id')
df_summary = df_summary[['class_id', 'pixel_count', 'area_ha', 'area_km2', 'pct_of_total']]
df_summary

In [ ]:
# quick bar chart of area by class
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df_summary['class_id'].astype(str), df_summary['area_ha'], color='steelblue')
ax.set_xlabel('Class ID')
ax.set_ylabel('Area (ha)')
ax.set_title('Land Cover Class Area — Final Classification')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# confidence stats per class (mean confidence for pixels assigned to each class)
confidence_by_class = max_confidence.addBands(final_lc).reduceRegion(
    reducer=ee.Reducer.mean().group(
        groupField=1,
        groupName='class'
    ),
    geometry=aoi,
    scale=100,
    maxPixels=1e13,
    bestEffort=True
).getInfo()

df_conf = pd.DataFrame(confidence_by_class['groups']).rename(
    columns={'class': 'class_id', 'mean': 'mean_confidence'}
)

df_summary_full = df_summary.merge(df_conf, on='class_id')
df_summary_full

# Post-classification refinement

In [ ]:
# load luma-ge predictors

from luma_ge.predictor import terrain_calculator, SpectralCalculator, distance_calculator
dist_calc = distance_calculator()
terrain_calc = terrain_calculator()
spectral_calc = SpectralCalculator()
#Define DEM
#retrieve topography based predictors
dem_source = 'NASADEM'
elevation = terrain_calc.calculate_elevation(aoi, dem_source=dem_source) # type: ignore
slope = terrain_calc.calculate_slope(aoi, dem_source=dem_source) # type: ignore
# aspect = terrain_calc.calculate_aspect(aoi, dem_source=dem_source) # type: ignore

#distance based predictors (5km buffer)
distance_predictor = dist_calc.calculate_distance_metrics(
    aoi = aoi, max_dist = 50000, in_meters = True)

In [ ]:
predictor_stack  = elevation.addBands(slope).addBands(distance_predictor).toFloat()
print('Final predictor stack bands:', predictor_stack.bandNames().getInfo() if predictor_stack else 'failed')

## Apply ruleset

Modified function from Faza to run fully on server side

In [ ]:
def build_validity_stack_ee(predictor_image, class_ids, ruleset_df, predictor_map=None):
    """
    Build a boolean validity mask per class, as a single multi-band ee.Image.

    Parameters
    ----------
    predictor_image : ee.Image
        Multi-band image containing the predictor bands referenced by
        predictor_map (e.g. 'elevation', distance bands, ...).
    class_ids : list
        Class_ID values, in the same order as prob_stack's bands.
    ruleset_df : pandas.DataFrame
        Indexed by Class_ID, with '<prefix>_min' / '<prefix>_max' columns.
    predictor_map : dict, optional
        Maps predictor band name -> rule column prefix. Defaults to
        DEFAULT_PREDICTOR_MAP.

    Returns
    -------
    ee.Image
        One band per class, named 'valid_<class_id>', in the input order.
        1 = pixel passes the rules for that class, 0 = it doesn't.
    """
    if predictor_map is None:
        predictor_map = DEFAULT_PREDICTOR_MAP

    predictor_band_names = predictor_image.bandNames().getInfo()

    valid_bands = []
    missing_classes = []

    for cid in class_ids:
        if cid not in ruleset_df.index:
            missing_classes.append(cid)
            valid_bands.append(ee.Image.constant(1).toByte().rename(f"valid_{cid}"))
            continue

        rule = ruleset_df.loc[cid]
        class_valid = ee.Image.constant(1).toByte()

        for band_name, prefix in predictor_map.items():
            if band_name not in predictor_band_names:
                continue
            min_col, max_col = f"{prefix}_min", f"{prefix}_max"
            if min_col not in rule or max_col not in rule:
                continue

            lo, hi = rule[min_col], rule[max_col]
            band = predictor_image.select(band_name)

            ok = ee.Image.constant(1).toByte()
            if pd.notna(lo):
                ok = ok.And(band.gte(float(lo)))
            if pd.notna(hi):
                ok = ok.And(band.lte(float(hi)))

            # masked / nodata predictor pixels are treated as valid,
            # mirroring the `ok |= np.isnan(vals)` behaviour of the original
            ok = ok.Or(band.mask().Not())

            class_valid = class_valid.And(ok)

        valid_bands.append(class_valid.rename(f"valid_{cid}"))

    if missing_classes:
        print(f"Warning: no ruleset row for IDs {sorted(set(missing_classes))}; unconstrained.")

    return ee.Image.cat(valid_bands)

def predict_class_ruleset_ee(valid_image, prob_stack, final_lc, max_confidence,
                              class_ids, aoi=None, scale=100, report_stats=True):
    """
    Argmax classification followed by rule-based post-processing, entirely
    server-side.

    Parameters
    ----------
    valid_image : ee.Image
        Output of build_validity_stack_ee — one 'valid_<id>' band per class,
        in the same order as prob_stack's bands.
    prob_stack : ee.Image
        Raw per-class probability image ('prob_<id>' bands), prob_stack from
        cell 3/5.
    final_lc : ee.Image
        Raw argmax classification (from cell 5), used as the fallback where
        no class is valid.
    max_confidence : ee.Image
        Raw argmax probability (from cell 5).
    class_ids : list
        Class_ID values in prob_stack band order (class_ids_list).
    aoi : ee.Geometry, optional
        Region to compute the "pixels changed" / "no valid class" counts
        over. Required only if report_stats=True.
    scale : int
        Scale (m) for the stats reduceRegion calls.
    report_stats : bool
        If True, triggers a couple of reduceRegion calls (like the original
        print statements) to report how many pixels changed / had no valid
        class. Set False to keep everything lazy/server-side.

    Returns
    -------
    dict with keys: classified_map, max_prob_map, raw_classified_map,
                     validity_stack, corrected_mask, no_valid_class_mask.
        All values are ee.Image objects (single band each, except
        validity_stack which keeps one band per class).
    """
    class_ids_list = list(class_ids)
    n = len(class_ids_list)

    # Large penalty pushes invalid classes below any real probability,
    # without disturbing relative ordering among valid classes.
    PENALTY = 1e6
    penalty = valid_image.subtract(1).multiply(PENALTY)  # 0 if valid=1, -PENALTY if valid=0
    penalty = penalty.rename(prob_stack.bandNames())      # align band-for-band with prob_stack

    adjusted_probs = prob_stack.add(penalty)
    adjusted_array = adjusted_probs.toArray()

    adjusted_argmax = adjusted_array.arrayArgmax().arrayGet(0)
    adjusted_maxprob = adjusted_array.arrayReduce(ee.Reducer.max(), [0]).arrayGet([0])

    adjusted_lc = adjusted_argmax.remap(
        ee.List.sequence(0, n - 1),
        ee.List(class_ids_list)
    ).rename("classification")

    # true where every class failed its rules for that pixel
    no_valid = valid_image.reduce(ee.Reducer.max()).eq(0).rename("no_valid_class")

    # fall back to the raw argmax/confidence where nothing passed the rules
    final_lc_corrected = final_lc.where(no_valid.eq(0), adjusted_lc).rename("classification")
    final_confidence = max_confidence.where(no_valid.eq(0), adjusted_maxprob).rename("confidence")

    # mask out pixels the raw classifier already had zero confidence for
    final_lc_corrected = final_lc_corrected.where(max_confidence.eq(0), 0)

    corrected = final_lc_corrected.neq(final_lc).And(max_confidence.neq(0)).rename("corrected")

    if report_stats:
        if aoi is None:
            print("report_stats=True but no aoi given; skipping pixel-count summary.")
        else:
            corrected_count = corrected.reduceRegion(
                reducer=ee.Reducer.sum(), geometry=aoi, scale=scale,
                maxPixels=1e13, bestEffort=True
            ).get("corrected").getInfo()
            no_valid_count = no_valid.reduceRegion(
                reducer=ee.Reducer.sum(), geometry=aoi, scale=scale,
                maxPixels=1e13, bestEffort=True
            ).get("no_valid_class").getInfo()
            print(f"Note: {int(no_valid_count or 0)} pixel(s) with no valid class; fallback to raw argmax.")
            print(f"Ruleset changed {int(corrected_count or 0)} pixels.")

    return {
        "classified_map": final_lc_corrected,
        "max_prob_map": final_confidence,
        "raw_classified_map": final_lc,
        "validity_stack": valid_image,
        "corrected_mask": corrected,
        "no_valid_class_mask": no_valid,
    }

In [ ]:
# BAND NAME MAPPING

DEFAULT_PREDICTOR_MAP = {
    "elevation": "dem",
    "slope": "slope",
    "dist_coast": "dist_coast",
    # "aceh_worldpop_2020_30m_ne": "worldpop",
}

ruleset_df = pd.read_excel('../data/modular_mapping_approach/sumatra_test/predictors_ruleset.xlsx', na_values=['-', ''])
ruleset_df = ruleset_df.set_index("Class_ID", drop=False)  

valid_image = build_validity_stack_ee(
    predictor_stack, class_ids_list, ruleset_df, DEFAULT_PREDICTOR_MAP
)

result = predict_class_ruleset_ee(
    valid_image, prob_stack, final_lc, max_confidence, class_ids_list,
    aoi=aoi, scale=100
)

final_lc_refined = result["classified_map"]
final_confidence_refined = result["max_prob_map"]

In [ ]:
raw_lc = result["raw_classified_map"]
corrected_lc = result["classified_map"]
corrected_mask = result["corrected_mask"]
no_valid_mask = result["no_valid_class_mask"]

# change-status image: 0=background, 1=unchanged, 2=corrected, 3=fallback (no valid class)
status = ee.Image(1).byte() \
    .where(raw_lc.eq(0), 0) \
    .where(corrected_mask, 2) \
    .where(no_valid_mask.And(raw_lc.neq(0)), 3) \
    .rename('status')

status_vis = {'min': 0, 'max': 3, 'palette': ['e0e0e0', 'd9f0d3', 'e31a1c', '6a3d9a']}

# --- Map 1: everything layered together ---
Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(raw_lc.clip(aoi), classification_vis, 'Raw Argmax')
Map.addLayer(corrected_lc.clip(aoi), classification_vis, 'Ruleset-Corrected')
Map.addLayer(status.clip(aoi), status_vis, 'Change Status')
# Map.add_legend(
#     title="Change status",
#     labels=["Background", "Unchanged", "Corrected", "Fallback"],
#     colors=["#e0e0e0", "#d9f0d3", "#e31a1c", "#6a3d9a"]
# )
Map

# --- Map 2: swipe comparison, raw vs corrected side by side ---

# swipe_map = geemap.Map()
# swipe_map.centerObject(aoi, 9)
# left_layer = geemap.ee_tile_layer(raw_lc.clip(aoi), classification_vis, 'Raw Argmax')
# right_layer = geemap.ee_tile_layer(corrected_lc.clip(aoi), classification_vis, 'Ruleset-Corrected')
# swipe_map.split_map(left_layer, right_layer)
# swipe_map

In [ ]:
# --- Stats: per-class area, raw vs corrected, computed server-side ---
raw_area = ee.Image.pixelArea().addBands(raw_lc).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
    geometry=aoi, scale=100, maxPixels=1e13, bestEffort=True
).getInfo()

corrected_area = ee.Image.pixelArea().addBands(corrected_lc).reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
    geometry=aoi, scale=100, maxPixels=1e13, bestEffort=True
).getInfo()

df_raw = pd.DataFrame(raw_area['groups']).rename(columns={'class': 'Class_ID', 'sum': 'area_m2_raw'})
df_corr = pd.DataFrame(corrected_area['groups']).rename(columns={'class': 'Class_ID', 'sum': 'area_m2_corrected'})

df_compare = df_raw.merge(df_corr, on='Class_ID', how='outer').fillna(0)
df_compare['area_ha_raw'] = df_compare['area_m2_raw'] / 10_000
df_compare['area_ha_corrected'] = df_compare['area_m2_corrected'] / 10_000
df_compare['change_ha'] = df_compare['area_ha_corrected'] - df_compare['area_ha_raw']
df_compare = df_compare.sort_values('Class_ID').reset_index(drop=True)
df_compare[['Class_ID', 'area_ha_raw', 'area_ha_corrected', 'change_ha']]

print(
    df_compare[
        ['Class_ID', 'area_ha_raw', 'area_ha_corrected', 'change_ha']
    ].to_string(index=False)
)

# Export final LULC map

In [ ]:
# final_corrected_stack = ee.Image.cat([
#     raw_lc,
#     corrected_lc,
#     corrected_mask,
#     no_valid_mask
# ])

# # Export to Earth Engine Asset
# task = ee.batch.Export.image.toDrive(
#     image=final_corrected_stack,
#     description=f'corrected_lc_Aceh_2020_{VERSION}',
#     folder='GEE_exports',
#     region=aoi,
#     scale=100,
#     maxPixels=1e13
# )

# task.start()